<div style="background-color: #f8fafc; border-left: 6px solid #2563eb; padding: 20px 24px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <span style="font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 1.5px; color: #2563eb;">AIML Honors &bull; Lab Assignment 01</span>
  <h1 style="margin: 8px 0 12px 0; color: #0f172a; font-size: 24px; font-weight: 700;">Multi-Gateway Payment Processing Engine</h1>
  <div style="font-size: 14px; color: #475569; line-height: 1.6;">
    <strong>Student Name:</strong> Pranjal Khadke &nbsp;|&nbsp; 
    <strong>Email:</strong> <a href="mailto:Pranjalckhadke@gmail.com" style="color: #2563eb; text-decoration: none;">Pranjalckhadke@gmail.com</a> &nbsp;|&nbsp; 
    <strong>GitHub:</strong> <a href="https://github.com/PranjalKhadke" style="color: #2563eb; text-decoration: none;">github.com/PranjalKhadke</a><br>
    <strong>Course:</strong> AIML Honors &bull; Sem III &nbsp;|&nbsp; 
    <strong>Topic:</strong> Object-Oriented Design & Factory Method Pattern
  </div>
</div>

### 1. Aim & Theoretical Foundations

**Aim:** To design and implement an extensible, secure multi-gateway payment processing engine utilizing Object-Oriented Programming (OOP) paradigms, specifically the **Factory Method Pattern**, **Polymorphism**, and **Data Abstraction** in Python.

#### Key Architectural Principles:
1. **Abstraction (`abc.ABC`):** Establishing a rigid interface contract (`PaymentMode` and `PaymentGateway`) ensuring that every concrete payment handler provides necessary billing and verification routines.
2. **Polymorphism:** The transaction switch treats any payment mechanism uniformly through `.process_charge()`, eliminating conditional type checks (`if type == ...`).
3. **Factory Method:** Delegating gateway instantiation to a registry factory (`get_gateway()`) so client code remains decoupled from internal gateway classes.

| Component | Responsibility | Concrete Classes |
| :--- | :--- | :--- |
| **`PaymentMode`** | Encapsulates user payment credentials & validation | `CardPayment`, `UPIPayment`, `NetBankingPayment` |
| **`PaymentGateway`** | Calculates processing fees, coordinates authorization | `RazorpayGateway`, `StripeGateway`, `PaytmGateway` |
| **`get_gateway()`** | Factory method instantiating gateways dynamically | Dictionary-based factory registry |


In [1]:
from abc import ABC, abstractmethod
import random

class PaymentMode(ABC):
    """Abstract base class for all payment modes."""
    @abstractmethod
    def get_summary(self):
        pass
        
    @abstractmethod
    def process_charge(self, total_amt, txn_id):
        pass

class PaymentGateway(ABC):
    """Abstract base class representing a payment aggregator."""
    def __init__(self, name, fee_rate, support_email):
        self.name = name
        self.fee_rate = fee_rate
        self.support_email = support_email
        
    def calculate_fee(self, amount):
        return round(amount * self.fee_rate, 2)
        
    def process_transaction(self, mode, amount):
        print("=" * 64)
        print(f"          {self.name.upper()} PAYMENT PROCESSING ENGINE")
        print("=" * 64)
        
        if amount <= 0:
            print(f"  [Rejection] Transaction declined: Amount (Rs. {amount}) must be > 0.")
            print("=" * 64 + "\n")
            return False
            
        txn_id = f"{self.name[:2].upper()}-TXN-{random.randint(100000, 999999)}"
        fee = self.calculate_fee(amount)
        final_bill = amount + fee
        
        print(f"  Transaction ID : {txn_id}")
        print(f"  Gateway Node   : {self.name} Central Switch")
        print(f"  Payment Mode   : {mode.get_summary()}")
        print("-" * 64)
        print(f"  Base Amount    : Rs. {amount:>12,.2f}")
        print(f"  Processing Fee : Rs. {fee:>12,.2f}  ({self.fee_rate * 100:.1f}%)")
        print(f"  Grand Total    : Rs. {final_bill:>12,.2f}")
        print("-" * 64)
        
        success = mode.process_charge(final_bill, txn_id)
        
        print("-" * 64)
        if success:
            print("  Transaction Status : SUCCESSFUL (HTTP 200 OK)")
        else:
            print("  Transaction Status : FAILED / REJECTED")
        print("=" * 64 + "\n")
        return success

In [2]:
class CardPayment(PaymentMode):
    def __init__(self, card_number, holder_name=""):
        clean_num = str(card_number).replace(" ", "").replace("-", "")
        self.card_number = clean_num
        self.holder_name = holder_name.strip()
        
    def get_summary(self):
        masked = f"****-****-****-{self.card_number[-4:]}" if len(self.card_number) >= 4 else self.card_number
        holder = f" [{self.holder_name}]" if self.holder_name else ""
        return f"Debit/Credit Card {masked}{holder}"
        
    def process_charge(self, total_amt, txn_id):
        if len(self.card_number) != 16 or not self.card_number.isdigit():
            print("  [Validation Error] Card number must contain exactly 16 numeric digits.")
            return False
        print(f"  [Card Network] Authorizing Rs. {total_amt:,.2f} on card {self.card_number[-4:]}...")
        print(f"  [Card Network] Payment authorized successfully under {txn_id}.")
        return True

class UPIPayment(PaymentMode):
    def __init__(self, vpa):
        self.vpa = vpa.strip()
        
    def get_summary(self):
        return f"Unified Payments Interface (UPI) [VPA: {self.vpa}]"
        
    def process_charge(self, total_amt, txn_id):
        if "@" not in self.vpa or self.vpa.startswith("@") or self.vpa.endswith("@"):
            print("  [Validation Error] Invalid UPI Virtual Payment Address format (expected user@psp).")
            return False
        print(f"  [UPI Gateway] Dispatching collect request of Rs. {total_amt:,.2f} to {self.vpa}...")
        print(f"  [UPI Gateway] PIN verified on device. Response code: 00 (Success).")
        return True

class NetBankingPayment(PaymentMode):
    def __init__(self, bank_name, user_id=""):
        self.bank_name = bank_name.strip()
        self.user_id = user_id.strip()
        
    def get_summary(self):
        uid = f" (User: {self.user_id})" if self.user_id else ""
        return f"Internet Banking [{self.bank_name}{uid}]"
        
    def process_charge(self, total_amt, txn_id):
        if not self.bank_name:
            print("  [Validation Error] Bank name cannot be empty for NetBanking.")
            return False
        print(f"  [Bank Gateway] Secure handshake with {self.bank_name} Core Banking Server...")
        print(f"  [Bank Gateway] 2FA OTP validated. Debited Rs. {total_amt:,.2f}.")
        return True

In [3]:
class RazorpayGateway(PaymentGateway):
    def __init__(self):
        super().__init__("Razorpay", 0.02, "support@razorpay.com")

class StripeGateway(PaymentGateway):
    def __init__(self):
        super().__init__("Stripe", 0.029, "support@stripe.com")

class PaytmGateway(PaymentGateway):
    def __init__(self):
        super().__init__("Paytm", 0.018, "care@paytm.com")

GATEWAY_MAP = {
    "razorpay": RazorpayGateway,
    "stripe": StripeGateway,
    "paytm": PaytmGateway,
}

def get_gateway(gateway_name):
    key = gateway_name.strip().lower()
    if key not in GATEWAY_MAP:
        supported = ", ".join(GATEWAY_MAP.keys())
        raise ValueError(f"Gateway '{gateway_name}' not configured. Options: {supported}")
    return GATEWAY_MAP[key]()

In [4]:
# Test Case 1: Processing UPI transaction via Razorpay
rp = get_gateway("razorpay")
upi_mode = UPIPayment("pranjal.khadke@oksbi")
rp.process_transaction(upi_mode, 1850.0)

          RAZORPAY PAYMENT PROCESSING ENGINE
  Transaction ID : RA-TXN-492108
  Gateway Node   : Razorpay Central Switch
  Payment Mode   : Unified Payments Interface (UPI) [VPA: pranjal.khadke@oksbi]
----------------------------------------------------------------
  Base Amount    : Rs.     1,850.00
  Processing Fee : Rs.        37.00  (2.0%)
  Grand Total    : Rs.     1,887.00
----------------------------------------------------------------
  [UPI Gateway] Dispatching collect request of Rs. 1,887.00 to pranjal.khadke@oksbi...
  [UPI Gateway] PIN verified on device. Response code: 00 (Success).
----------------------------------------------------------------
  Transaction Status : SUCCESSFUL (HTTP 200 OK)



In [5]:
# Test Case 2: Processing Credit Card transaction via Stripe
stripe_gw = get_gateway("stripe")
card_mode = CardPayment("4532789012345678", "Pranjal Khadke")
stripe_gw.process_transaction(card_mode, 4750.0)

          STRIPE PAYMENT PROCESSING ENGINE
  Transaction ID : ST-TXN-831940
  Gateway Node   : Stripe Central Switch
  Payment Mode   : Debit/Credit Card ****-****-****-5678 [Pranjal Khadke]
----------------------------------------------------------------
  Base Amount    : Rs.     4,750.00
  Processing Fee : Rs.       137.75  (2.9%)
  Grand Total    : Rs.     4,887.75
----------------------------------------------------------------
  [Card Network] Authorizing Rs. 4,887.75 on card 5678...
  [Card Network] Payment authorized successfully under ST-TXN-831940.
----------------------------------------------------------------
  Transaction Status : SUCCESSFUL (HTTP 200 OK)



In [6]:
# Test Case 3: Processing NetBanking transaction via Paytm
paytm_gw = get_gateway("paytm")
nb_mode = NetBankingPayment("HDFC Bank", "CUST_7891")
paytm_gw.process_transaction(nb_mode, 920.0)

          PAYTM PAYMENT PROCESSING ENGINE
  Transaction ID : PA-TXN-159234
  Gateway Node   : Paytm Central Switch
  Payment Mode   : Internet Banking [HDFC Bank (User: CUST_7891)]
----------------------------------------------------------------
  Base Amount    : Rs.       920.00
  Processing Fee : Rs.        16.56  (1.8%)
  Grand Total    : Rs.       936.56
----------------------------------------------------------------
  [Bank Gateway] Secure handshake with HDFC Bank Core Banking Server...
  [Bank Gateway] 2FA OTP validated. Debited Rs. 936.56.
----------------------------------------------------------------
  Transaction Status : SUCCESSFUL (HTTP 200 OK)



In [7]:
# Test Case 4: Exception & validation handling
bad_card = CardPayment("1234", "Invalid Test")
rp.process_transaction(bad_card, 500.0)

rp.process_transaction(card_mode, -250.0)

          RAZORPAY PAYMENT PROCESSING ENGINE
  Transaction ID : RA-TXN-902341
  Gateway Node   : Razorpay Central Switch
  Payment Mode   : Debit/Credit Card 1234
----------------------------------------------------------------
  Base Amount    : Rs.       500.00
  Processing Fee : Rs.        10.00  (2.0%)
  Grand Total    : Rs.       510.00
----------------------------------------------------------------
  [Validation Error] Card number must contain exactly 16 numeric digits.
----------------------------------------------------------------
  Transaction Status : FAILED / REJECTED

          RAZORPAY PAYMENT PROCESSING ENGINE
  [Rejection] Transaction declined: Amount (Rs. -250.0) must be > 0.



<div style="background-color: #f0fdf4; border: 1px solid #bbf7d0; border-left: 5px solid #16a34a; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">
  <h4 style="margin: 0 0 6px 0; color: #15803d; font-size: 15px; font-weight: 700;">Conclusion & Key Takeaways</h4>
  <p style="margin: 0; color: #166534; font-size: 13.5px; line-height: 1.6;">
    The implementation successfully decouples payment method authorization from gateway fee accounting. Adding new payment modes (e.g., Crypto, Wallets) or additional gateways (e.g., PayU, CCAvenue) requires no modifications to the central execution switch, adhering strictly to the <strong>Open/Closed Principle (SOLID)</strong>. Input validation guards successfully intercept malformed credentials and negative amounts.
  </p>
</div>